# AI Security Questionnaire Responder — demo
**Phase 1:** one question end to end, with citations and a measured confidence level (sections 1–12).  
**Phase 2:** a whole questionnaire in parallel, written back to XLSX with an audit trail (sections 13–18).

We're the vendor, **Acme SaaS** (fictional). A prospect has sent a security / AI due-diligence questionnaire. This notebook shows the core loop for a single question:

```
question → classify → specialist agent (searches the policy pack) → grounding check → confidence score
```

- **Retrieval:** Acme's policy pack, chopped into labelled sections (e.g. `AIG-3`) and indexed in SQLite FTS5.
- **Confidence** is calculated from measured signals (are the claims backed by quotes from the cited sections? how strong was the search match? is it a restricted topic?), **not** the model's opinion of itself.
- If the docs don't cover it, the answer is **"Not covered by policy, escalate"**, never a guess.

**Before you start:** add `ANTHROPIC_API_KEY` in Colab Secrets (🔑 in the left sidebar), and have `acme-policy-pack.zip` ready to upload.

## 1. Setup

In [ ]:
!pip install -q langgraph langchain-anthropic

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

SPECIALIST_MODEL = "claude-opus-5-5"   # specialist agents + grounding check
CLASSIFIER_MODEL = "claude-haiku-5-5"  # cheap routing step (set to SPECIALIST_MODEL if it errors)

# Confidence thresholds. Placeholders for now; Phase 4 evals will calibrate them.
THRESHOLDS = {
    "high_min_support": 1.0,      # share of claims backed by a cited section
    "high_min_explicit": 1.0,     # share of claims with an exact quote found in the cited section
    "medium_min_support": 0.75,   # below this the answer goes to Low
    "min_retrieval_score": 4.0,   # weakest search score (BM25) for High; trap questions score ~3-4, real answers 8+
}
AUTO_APPROVE_LEVELS = {"High"}    # anything else goes to a human
ESCALATE_TEXT = "Not covered by policy, escalate"
print("Config loaded.")

## 2. Load the Acme SaaS policy pack
Upload `acme-policy-pack.zip` when prompted. (If the files are already in the Colab session, this skips the upload.)

In [ ]:
import glob, pathlib, zipfile

if not glob.glob("**/data/policies/*.md", recursive=True):
    from google.colab import files
    uploaded = files.upload()                      # choose acme-policy-pack.zip
    for name in uploaded:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(".")

POLICY_DIR = pathlib.Path(sorted(glob.glob("**/data/policies", recursive=True))[0])
policy_files = sorted(POLICY_DIR.glob("*.md"))
print(f"{len(policy_files)} policy docs in {POLICY_DIR}:")
for p in policy_files:
    print("  -", p.name)

## 3. Chop the docs into sections
Each `## ISP-6 Encryption`-style heading becomes one searchable passage. The section ID is what answers cite.

In [ ]:
import re

def parse_doc(path):
    text = path.read_text(encoding="utf-8")
    meta = {}
    m = re.match(r"^---\n(.*?)\n---\n", text, re.S)          # front matter
    if m:
        for line in m.group(1).splitlines():
            key, _, value = line.partition(":")
            meta[key.strip()] = value.strip()
        text = text[m.end():]
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        head = re.match(r"## ([A-Z]+-\d+) (.+)", section)
        if not head:
            continue
        chunks.append({
            "section_id": head.group(1),
            "heading": head.group(2).strip(),
            "doc_id": meta.get("doc_id", ""),
            "doc_title": meta.get("title", path.stem),
            "version": meta.get("version", ""),
            "text": section[head.end():].strip(),
        })
    return chunks

chunks = [c for p in policy_files for c in parse_doc(p)]
print(f"{len(chunks)} sections. Example:\n")
print(chunks[0]["section_id"], "|", chunks[0]["doc_title"], "|", chunks[0]["heading"])
print(chunks[0]["text"])

## 4. Index them in SQLite FTS5
FTS5 is SQLite's full-text search, like the index at the back of a book. `porter` stemming means *encrypt* also matches *encrypted*. Results are ranked with BM25 (higher = stronger match), and headings count double.

In [ ]:
import sqlite3

DB_PATH = "acme_policies.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute('''
    CREATE VIRTUAL TABLE passages USING fts5(
        section_id UNINDEXED, doc_id UNINDEXED, doc_title UNINDEXED, version UNINDEXED,
        heading, text,
        tokenize = 'porter unicode61'
    )''')
conn.executemany(
    "INSERT INTO passages VALUES (:section_id, :doc_id, :doc_title, :version, :heading, :text)",
    chunks)
conn.commit()

STOPWORDS = set('''a an and are as at be by can do does for from has have how i if in
into is it its of on or our please provide the their there this to us we what when
where which who will with you your yours any all describe detail details'''.split())

def to_fts_query(text):
    terms = [t for t in re.findall(r"[a-z0-9]+", text.lower())
             if t not in STOPWORDS and len(t) > 1]
    return " OR ".join(f'"{t}"' for t in dict.fromkeys(terms))   # quoted = safe from FTS syntax

def search(query, k=4):
    fts = to_fts_query(query)
    if not fts:
        return []
    rows = conn.execute('''
        SELECT section_id, doc_title, heading, text,
               -bm25(passages, 0, 0, 0, 0, 2.0, 1.0) AS score
        FROM passages WHERE passages MATCH ?
        ORDER BY score DESC LIMIT ?''', (fts, k)).fetchall()
    return [dict(r) for r in rows]

def format_hits(hits):
    return "\n\n".join(
        f"[{h['section_id']}] {h['doc_title']}: {h['heading']} (score {h['score']:.2f})\n{h['text']}"
        for h in hits)

print(f"Indexed {conn.execute('SELECT count(*) FROM passages').fetchone()[0]} sections.\n")
for q in ["Do you train AI models on customer data?", "Is data encrypted at rest?", "bug bounty programme"]:
    print(f"Q: {q}")
    for h in search(q, k=3):
        print(f"   {h['score']:5.2f}  {h['section_id']:7} {h['heading']}")
    print()

## 5. Structured outputs and graph state
Every model step returns a typed object, so later steps (and the audit trail) can rely on its shape.

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field

class Classification(BaseModel):
    category: Literal["security", "ai_governance", "privacy"]
    asks_for_commitment: bool = Field(description=(
        "True ONLY if the question asks the vendor to go beyond describing its current practice: "
        "to guarantee or warrant an outcome, agree new or different contractual terms, accept "
        "liability or indemnity, agree SLA penalties or service credits, or hand over a confidential "
        "report. False for questions asking what the vendor's existing policy, timeframe, notice "
        "period or process is, even when the answer describes something the vendor already does."))
    reason: str = Field(description="One short sentence explaining the category.")

class DraftAnswer(BaseModel):
    covered: bool = Field(description="False if the passages do not answer the question.")
    answer: str = Field(description="The answer as the vendor would write it. Empty if not covered.")
    cited_section_ids: list[str] = Field(description="Section IDs the answer relies on, e.g. ['AIG-3'].")

class Claim(BaseModel):
    claim: str = Field(description="One factual statement taken from the answer.")
    cited_section_id: str = Field(description="The one section that supports it, or '' if none.")
    evidence_quote: str = Field(description="Exact words copied from that section that support the claim, or ''.")
    verdict: Literal["explicit", "inferred", "unsupported"] = Field(description=(
        "explicit: the section states it directly; inferred: it follows from the section but is not "
        "stated; unsupported: no cited section supports it."))

class GroundingReport(BaseModel):
    claims: list[Claim]

class QState(TypedDict, total=False):
    question: str
    classification: dict
    restricted: bool
    restricted_reasons: list[str]
    search_log: list[str]
    passages: dict        # section_id -> passage dict
    draft: dict
    grounding: dict
    result: dict

print("Schemas ready.")

## 6. The models and prompts

In [ ]:
from langchain_anthropic import ChatAnthropic

classifier_llm = ChatAnthropic(model=CLASSIFIER_MODEL, max_tokens=1024, max_retries=3)
specialist_llm = ChatAnthropic(model=SPECIALIST_MODEL, max_tokens=2048, max_retries=3)

CLASSIFIER_PROMPT = '''You route questions from a customer security / AI due-diligence questionnaire
to the right specialist at a SaaS vendor.
- security: infosec programme, certifications, access control, encryption, vulnerability management,
  hosting, network, identity/SSO, incident response, business continuity.
- ai_governance: AI features, models, model providers, training on data, AI risk management,
  AI regulation (EU AI Act, NIST AI RMF, ISO 42001), AI testing and oversight.
- privacy: personal data, GDPR, DPAs, data retention and deletion, sub-processors, data transfers,
  data subject rights.'''

COMMON_RULES = '''You work for Acme SaaS (the vendor) and answer questions from a prospect's
due-diligence questionnaire using ONLY Acme's internal policy pack.
Rules:
- Every statement must come from a retrieved policy section. Never use outside knowledge about
  Acme or assume typical industry practice.
- If the sections do not answer the question, say so. Do not guess or fill gaps.
- Do not add guarantees, promises or commitments that the policy does not state.'''

SPECIALIST_FOCUS = {
    "security": "You are Acme's security specialist. Be precise about certifications, scopes, "
                "time frames and technical controls.",
    "ai_governance": "You are Acme's AI governance specialist. Be precise about how customer data "
                     "is (and is not) used by AI, model providers, human oversight and regulatory "
                     "positioning. Never overstate certifications that are only in progress.",
    "privacy": "You are Acme's privacy specialist. Be precise about controller/processor roles, "
               "retention periods, deletion, sub-processors and data locations.",
}

DRAFT_PROMPT = '''Write the questionnaire answer from the passages provided.
- Answer exactly what was asked, in 1 to 3 sentences of plain professional English, in Acme's voice ("Acme ...", "We ...").
- Include the specifics the question asks for (numbers, time frames, standards). Do not add related
  facts the question didn't ask about: every extra sentence is one more claim a reviewer must check.
- For a yes/no question, start with "Yes." or "No." only when a passage states it directly.
- Do not put section IDs in the answer text; list them in cited_section_ids.
- If the passages do not answer the question, set covered=false, answer="" and cited_section_ids=[].'''

GROUNDING_PROMPT = '''You are an auditor checking a questionnaire answer against its cited policy sections.
Split the answer into its individual factual claims. A bare "Yes." or "No." is not a separate claim:
treat it as part of the claim that follows. For each claim, pick the one cited section that supports
it and copy the exact supporting words from that section into evidence_quote (verbatim, no
paraphrasing). Verdicts:
- explicit: the quoted words state the same fact, even if the answer words it differently;
- inferred: the claim adds meaning the quote does not contain (a conclusion, an absence, a wider scope);
- unsupported: no cited section supports it.
Be strict about meaning, not wording: anything not in the cited sections is unsupported.'''
print("Models and prompts ready.")

## 7. Graph nodes
- **classify:** picks the specialist and checks for restricted topics, using a keyword rule *and* the classifier.
- **specialist:** an agent with a `search_policies` tool. It runs its own searches, then drafts an answer from what it found.
- **grounding_check:** splits the answer into claims and checks each one. The code then confirms each quote really appears in the cited section; an explicit claim whose quote can't be found is downgraded.
- **score:** turns those measurements into High / Medium / Low.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

RESTRICTED_RULES = {
    "guarantee":           r"\bguarantee",
    "warranty":            r"\bwarrant",
    "liability":           r"\bliabilit|\bliable\b",
    "indemnity":           r"\bindemn",
    "commitment":          r"\bcommit(ment)? to\b|\bcontractual|\bin (the|our|your) contract\b",
    "sla / credits":       r"\bSLA\b|service credit|\bpenalt",
    "confidential report": r"\b(full|complete) (pen(etration)? test|audit|soc ?2) report",
}

def classify(state: QState) -> dict:
    q = state["question"]
    c = classifier_llm.with_structured_output(Classification, method="json_schema").invoke(
        [SystemMessage(CLASSIFIER_PROMPT), HumanMessage(q)])
    reasons = [f"keyword rule: {name}" for name, rx in RESTRICTED_RULES.items() if re.search(rx, q, re.I)]
    if c.asks_for_commitment:
        reasons.append(f"classifier: asks for a commitment ({c.reason})")
    return {"classification": c.model_dump(), "restricted": bool(reasons), "restricted_reasons": reasons}


MAX_TOOL_ROUNDS = 4

def specialist(state: QState) -> dict:
    q, category = state["question"], state["classification"]["category"]
    found, log = {}, []

    def remember(hits):
        for h in hits:
            if h["section_id"] not in found or h["score"] > found[h["section_id"]]["score"]:
                found[h["section_id"]] = h

    @tool
    def search_policies(query: str) -> str:
        '''Search Acme SaaS's internal policy pack. Use short keyword queries
        (e.g. "encryption at rest keys"). Returns matching sections with their IDs.'''
        hits = search(query)
        log.append(query)
        remember(hits)
        return format_hits(hits) or "No matching sections."

    remember(search(q))                     # baseline search on the raw question
    log.append(f"(question) {q}")

    agent = specialist_llm.bind_tools([search_policies])
    messages = [
        SystemMessage(COMMON_RULES + "\n\n" + SPECIALIST_FOCUS[category]),
        HumanMessage(f"Questionnaire question: {q}\n\nSearch the policy pack with 1-3 different "
                     "keyword queries to find every relevant section. When you have enough, reply DONE."),
    ]
    for _ in range(MAX_TOOL_ROUNDS):
        ai = agent.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            break
        for tc in ai.tool_calls:
            messages.append(ToolMessage(search_policies.invoke(tc["args"]), tool_call_id=tc["id"]))

    if not found:
        return {"passages": {}, "search_log": log,
                "draft": {"covered": False, "answer": "", "cited_section_ids": []}}

    ranked = sorted(found.values(), key=lambda h: -h["score"])
    draft = specialist_llm.with_structured_output(DraftAnswer, method="json_schema").invoke([
        SystemMessage(COMMON_RULES + "\n\n" + SPECIALIST_FOCUS[category] + "\n\n" + DRAFT_PROMPT),
        HumanMessage(f"Question: {q}\n\nPolicy sections:\n\n{format_hits(ranked)}"),
    ])
    d = draft.model_dump()
    d["cited_section_ids"] = [s for s in dict.fromkeys(d["cited_section_ids"]) if s in found]  # only real, retrieved IDs
    return {"passages": found, "search_log": log, "draft": d}


def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s%.]", " ", s.lower())).strip()

def grounding_check(state: QState) -> dict:
    d = state["draft"]
    if not d["covered"] or not d["cited_section_ids"]:
        return {"grounding": {"claims": []}}
    cited = {s: state["passages"][s] for s in d["cited_section_ids"]}
    report = specialist_llm.with_structured_output(GroundingReport, method="json_schema").invoke([
        SystemMessage(GROUNDING_PROMPT),
        HumanMessage(f"Question: {state['question']}\n\nAnswer: {d['answer']}\n\n"
                     f"Cited sections:\n\n{format_hits(list(cited.values()))}"),
    ])
    claims = []
    for c in report.claims:
        c = c.model_dump()
        section = cited.get(c["cited_section_id"])
        c["quote_found"] = bool(section and c["evidence_quote"].strip()
                                and _norm(c["evidence_quote"]) in _norm(section["text"]))
        if c["verdict"] != "unsupported" and section is None:
            c["verdict"], c["note"] = "unsupported", "cited a section that wasn't retrieved"
        elif c["verdict"] == "explicit" and not c["quote_found"]:
            c["verdict"], c["note"] = "inferred", "quote not found verbatim; downgraded"
        claims.append(c)
    return {"grounding": {"claims": claims}}


def score(state: QState) -> dict:
    d, claims = state["draft"], state["grounding"]["claims"]
    n = len(claims)
    support = sum(c["verdict"] != "unsupported" for c in claims) / n if n else 0.0
    explicit = sum(c["verdict"] == "explicit" for c in claims) / n if n else 0.0
    best = max((state["passages"][s]["score"] for s in d["cited_section_ids"]), default=0.0)
    signals = {
        "library_match": False,                      # Phase 3: approved-answer library
        "best_retrieval_score": round(best, 2),
        "claims": n,
        "claim_support": round(support, 2),
        "explicit_evidence": round(explicit, 2),
        "restricted": state["restricted"],
    }
    reasons, answer = [], d["answer"]

    if not d["covered"] or n == 0 or support == 0:
        level, answer = "Low", ESCALATE_TEXT
        reasons.append("policy pack doesn't cover this question")
    elif (support >= THRESHOLDS["high_min_support"] and explicit >= THRESHOLDS["high_min_explicit"]
          and best >= THRESHOLDS["min_retrieval_score"]):
        level = "High"
        reasons.append("every claim is backed by a verbatim quote from a cited section")
    elif support >= THRESHOLDS["medium_min_support"]:
        level = "Medium"
        if explicit < THRESHOLDS["high_min_explicit"]:
            reasons.append("some claims are inferred rather than stated")
        if best < THRESHOLDS["min_retrieval_score"]:
            reasons.append("weak search match")
        if support < 1:
            reasons.append("some claims are unsupported")
    else:
        level = "Low"
        reasons.append(f"only {support:.0%} of claims are supported")

    if state["restricted"]:
        level = "Low"
        reasons += [f"restricted topic, {r}" for r in state["restricted_reasons"]]

    status = ("escalate" if answer == ESCALATE_TEXT
              else "auto_approve" if level in AUTO_APPROVE_LEVELS
              else "human_review")
    return {"result": {"question": state["question"], "category": state["classification"]["category"],
                       "answer": answer, "draft_answer": d["answer"],
                       "sources": d["cited_section_ids"] if answer != ESCALATE_TEXT else [],
                       "confidence": level, "status": status, "reasons": reasons, "signals": signals}}

print("Nodes defined.")

## 8. Wire up the graph

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(QState)
builder.add_node("classify", classify)
builder.add_node("specialist", specialist)
builder.add_node("grounding_check", grounding_check)
builder.add_node("score", score)
builder.add_edge(START, "classify")
builder.add_edge("classify", "specialist")
builder.add_edge("specialist", "grounding_check")
builder.add_edge("grounding_check", "score")
builder.add_edge("score", END)
graph = builder.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 9. Answer a question

In [ ]:
ICON = {"High": "🟢", "Medium": "🟡", "Low": "🔴"}

def show(state):
    r = state["result"]
    print(f"Q: {r['question']}")
    print(f"Specialist: {r['category']}\n")
    print(f"A: {r['answer']}")
    if r["answer"] == ESCALATE_TEXT and r["draft_answer"]:
        print(f"   (draft for reviewer: {r['draft_answer']})")
    print(f"\nSources: {', '.join(r['sources']) or 'none'}")
    for s in r["sources"]:
        p = state["passages"][s]
        print(f"   [{s}] {p['doc_title']}: {p['heading']}")
    print(f"\nConfidence: {ICON[r['confidence']]} {r['confidence']}   Status: {r['status']}")
    for why in r["reasons"]:
        print(f"   - {why}")
    print(f"Signals: {r['signals']}")
    print(f"Searches: {state['search_log']}")
    if state["grounding"]["claims"]:
        print("\nClaim check:")
        for c in state["grounding"]["claims"]:
            mark = {"explicit": "✓", "inferred": "~", "unsupported": "✗"}[c["verdict"]]
            print(f"   {mark} [{c['cited_section_id'] or '-'}] {c['claim']}")
            if c.get("note"):
                print(f"       ({c['note']})")

def ask(question):
    state = graph.invoke({"question": question})
    show(state)
    return state

s1 = ask("Do you use customer data to train or fine-tune AI models?")

## 10. A trap question
The policy pack deliberately says nothing about a bug bounty. The right behaviour is to escalate, not invent one.

In [ ]:
s2 = ask("Do you operate a public bug bounty programme?")

## 11. A restricted question
The docs *do* mention availability (99.9% target), but the prospect wants a contractual guarantee. That's a human decision, so it goes to review whatever the evidence says.

In [ ]:
s3 = ask("Can you guarantee 99.99% uptime with service credits?")

## 12. Try your own

In [ ]:
ask("Is customer data encrypted at rest, and who manages the keys?");

---
# Phase 2 — Answer a whole questionnaire
These sections reuse the policy pack, search index and graph from sections 1–8 (sections 9–12 are optional).

## 13. Upload the questionnaire
Any XLSX or CSV. You can also upload `golden_set.csv` at the same time, for the answer-key check in section 18. If you upload nothing, it uses a questionnaire from the pack if there is one.

In [ ]:
from google.colab import files

print("Choose the questionnaire (XLSX or CSV), plus golden_set.csv if you have it.")
try:
    uploaded = files.upload()
except Exception:
    uploaded = {}
candidates = [n for n in uploaded if not n.startswith("golden_set")]
if candidates:
    QFILE = pathlib.Path(candidates[0])
else:
    QFILE = pathlib.Path(sorted(glob.glob("**/data/questionnaires/*.xlsx", recursive=True))[0])
print("Questionnaire:", QFILE)

## 14. Find the questions
Real questionnaires are messy, so nothing here is hard-coded to one layout. On each tab it:
- finds the **header row** (a "Question" column plus an "Answer / Response / Comments" column), wherever it sits;
- works out which columns hold the **ref**, the **question**, the **answer** and any **evidence** field;
- skips cover pages, section-header rows, blank rows, footers and reviewer-only columns;
- gives sub-questions like `a) …` the text of their parent row, so they make sense on their own;
- skips questions that already have an answer.

In [ ]:
import csv, openpyxl

Q_HDR   = re.compile(r"^\s*(question|questions|question text|requirement|control question)\b", re.I)
REF_HDR = re.compile(r"^\s*(ref\.?|reference|q\s*#|q\s*no\.?|#|id|no\.?|item|question id|control id)\s*$", re.I)
ANS_HDRS = [r"supplier response", r"vendor response", r"\banswer\b", r"supporting comments",
            r"\bcomments?\b", r"\bresponse\b"]                    # first match wins
CHOICE_HDR = re.compile(r"^\s*(response|yes\s*/\s*no|y\s*/\s*n|compliant\??)\s*$", re.I)
EVID_HDR = re.compile(r"evidence|supporting doc", re.I)
NOT_OURS = re.compile(r"reviewer|internal|office use|assessor|score|weight", re.I)   # columns we must not fill
SUBQ = re.compile(r"^\s*\(?([a-h]|i{1,3}|iv|v)\)\s+")

def _txt(v):
    return "" if v is None else str(v).strip()

def read_grid(path):
    # -> {sheet: {"rows": {r: [values]}, "merged": {r: text}}}; merged = rows merged across columns
    if path.suffix.lower() == ".csv":
        with open(path, newline="", encoding="utf-8-sig") as f:
            rows = list(csv.reader(f))
        return {"csv": {"rows": {i + 1: r for i, r in enumerate(rows)}, "merged": {}}}
    wb = openpyxl.load_workbook(path)
    out = {}
    for ws in wb.worksheets:
        if ws.sheet_state != "visible":
            continue
        rows = {r: [c.value for c in row] for r, row in enumerate(ws.iter_rows(), start=1)}
        merged = {}
        for rng in ws.merged_cells.ranges:
            if rng.max_col > rng.min_col:
                for r in range(rng.min_row, rng.max_row + 1):
                    merged[r] = _txt(ws.cell(rng.min_row, rng.min_col).value)
        out[ws.title] = {"rows": rows, "merged": merged}
    return out

def find_layout(rows, scan=25):
    for r in sorted(rows)[:scan]:
        hdr = {c: _txt(v) for c, v in enumerate(rows[r]) if _txt(v)}
        q_col = next((c for c, h in hdr.items() if Q_HDR.search(h) and not NOT_OURS.search(h)), None)
        if q_col is None:
            continue
        ans_col = None
        for pat in ANS_HDRS:
            ans_col = next((c for c, h in hdr.items() if c != q_col and re.search(pat, h, re.I)
                            and not NOT_OURS.search(h)), None)
            if ans_col is not None:
                break
        if ans_col is None:
            continue
        ref_col = next((c for c, h in hdr.items() if REF_HDR.match(h)), None)
        choice_col = next((c for c, h in hdr.items() if c not in (ans_col, q_col) and CHOICE_HDR.match(h)), None)
        evid_col = next((c for c, h in hdr.items() if c != ans_col and EVID_HDR.search(h)
                         and not NOT_OURS.search(h)), None)
        return {"header_row": r, "q": q_col, "ref": ref_col, "answer": ans_col,
                "choice": choice_col, "evidence": evid_col, "headers": hdr}
    return None

def extract_questions(path):
    items, report = [], []
    for sheet, g in read_grid(path).items():
        rows, merged = g["rows"], g["merged"]
        lay = find_layout(rows)
        if not lay:
            report.append(f"  {sheet}: no question table found, skipped")
            continue
        h = lay["headers"]
        report.append(f"  {sheet}: header row {lay['header_row']} | question='{h[lay['q']]}' "
                      f"answer='{h[lay['answer']]}'"
                      + (f" ref='{h[lay['ref']]}'" if lay["ref"] is not None else "")
                      + (f" evidence='{h[lay['evidence']]}'" if lay["evidence"] is not None else ""))
        section, n, prefilled = "", 0, 0
        for r in sorted(rows):
            if r <= lay["header_row"]:
                continue
            vals = list(rows[r]) + [None] * 30
            if r in merged:                                   # section header / footer
                section = merged[r]
                continue
            q = _txt(vals[lay["q"]])
            if len(q) < 10 or not re.search(r"[A-Za-z]{3}", q):
                continue
            if _txt(vals[lay["answer"]]):
                prefilled += 1
                continue
            ref = _txt(vals[lay["ref"]]) if lay["ref"] is not None else ""
            model_q = re.sub(r"\s+", " ", q)
            if SUBQ.match(q) and section:                     # a) b) c) -> add parent context
                parent = re.sub(r"^[\d.]+\s*", "", section)
                model_q = f"{parent}: {SUBQ.sub('', model_q)}"
            items.append({"sheet": sheet, "row": r, "ref": ref or f"{sheet}!{r}", "question": q,
                          "model_question": model_q, "section": section,
                          "answer_col": lay["answer"], "evidence_col": lay["evidence"]})
            n += 1
        report[-1] += f"\n      -> {n} questions" + (f", {prefilled} already answered (skipped)" if prefilled else "")
    return items, report

items, report = extract_questions(QFILE)
print(f"{QFILE.name}:")
print("\n".join(report))
print(f"\n{len(items)} questions to answer. First few:")
for it in items[:3] + [i for i in items if i["model_question"] != re.sub(r'\s+', ' ', i["question"])][:2]:
    print(f"  [{it['sheet']} {it['ref']}] {it['model_question'][:110]}")

## 15. Answer them all in parallel
LangGraph's **`Send`** API fans out one task per question. Each runs the same classify → specialist → grounding check → score graph you tested above, several at a time, and the results are collected back into one list.

`MAX_PARALLEL` caps how many questions run at once, to stay within API rate limits. If one question fails it's marked as an error, and the rest carry on.

In [ ]:
import operator, threading, time
from typing import Annotated
from langgraph.types import Send

MAX_PARALLEL = 6

# SQLite connections aren't safe to share across threads without a lock
_search_lock = threading.Lock()
if "_search_unlocked" not in globals():
    _search_unlocked = search
def search(query, k=4):
    with _search_lock:
        return _search_unlocked(query, k)

class BatchState(TypedDict):
    items: list
    results: Annotated[list, operator.add]     # each parallel task appends its result

def fan_out(state: BatchState):
    return [Send("answer_one", item) for item in state["items"]]

def answer_one(item: dict) -> dict:
    t0 = time.time()
    try:
        s = graph.invoke({"question": item["model_question"]})
        r = dict(s["result"])
        r["claims"] = s["grounding"]["claims"]
        r["search_log"] = s["search_log"]
        r["source_titles"] = [f"{s['passages'][sid]['doc_title']} §{sid}" for sid in r["sources"]]
    except Exception as e:
        r = {"answer": "", "draft_answer": "", "sources": [], "source_titles": [], "category": "",
             "confidence": "Low", "status": "error", "reasons": [f"{type(e).__name__}: {e}"],
             "signals": {}, "claims": [], "search_log": []}
    r["seconds"] = round(time.time() - t0, 1)
    return {"results": [{**r, **item}]}        # item keys (sheet, ref, original question) win

batch_builder = StateGraph(BatchState)
batch_builder.add_node("answer_one", answer_one)
batch_builder.add_conditional_edges(START, fan_out, ["answer_one"])
batch_builder.add_edge("answer_one", END)
batch_graph = batch_builder.compile()

results, t0 = [], time.time()
print(f"Answering {len(items)} questions, {MAX_PARALLEL} at a time...\n")
for update in batch_graph.stream({"items": items, "results": []},
                                 config={"max_concurrency": MAX_PARALLEL}, stream_mode="updates"):
    for r in update.get("answer_one", {}).get("results", []):
        results.append(r)
        icon = {"auto_approve": "🟢", "human_review": "🟡", "escalate": "🔴", "error": "⚠️"}[r["status"]]
        print(f"{len(results):3}/{len(items)} {icon} {r['status']:12} [{r['sheet'][:18]:18} {r['ref']:7}] {r['question'][:60]}")
order = {(i["sheet"], i["row"]): n for n, i in enumerate(items)}
results.sort(key=lambda r: order[(r["sheet"], r["row"])])
print(f"\nDone in {time.time() - t0:.0f}s.")

## 16. Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(df["status"].value_counts().to_string(), "\n")
print(pd.crosstab(df["sheet"], df["status"]).to_string(), "\n")
print("Auto-answer rate:", f"{(df['status'] == 'auto_approve').mean():.0%}")
errors = df[df["status"] == "error"]
if len(errors):
    print(f"\n⚠️ {len(errors)} errors, e.g.: {errors.iloc[0]['reasons'][0][:200]}")
pd.set_option("display.max_colwidth", 80)
df[["sheet", "ref", "question", "category", "confidence", "status", "sources"]]

## 17. Write the completed questionnaire + audit report
- **The prospect's file**, with answers written into their own answer cells and all their formatting, dropdowns and tabs kept. Cells are colour-coded for your review: 🟩 auto-approved, 🟧 `[REVIEW]` draft needing a human, 🟥 `[ESCALATE]` not covered by policy. Evidence columns get the cited sections.
- **A `Responder Summary` tab**, added at the end, with one row per question: confidence, status, sources, reasons and claim counts.
- **An audit CSV**, with the full trail per question: searches run, every claim and its verdict, and the signals behind the confidence level.

In [ ]:
import json
from openpyxl.styles import PatternFill, Font, Alignment

FILL = {"auto_approve": "E2EFDA", "human_review": "FCE4D6", "escalate": "F8CBAD", "error": "D9D9D9"}

def cell_text(r):
    if r["status"] == "auto_approve":
        return r["answer"]
    if r["status"] == "human_review":
        return "[REVIEW] " + r["draft_answer"]
    if r["status"] == "escalate":
        return "[ESCALATE] " + ESCALATE_TEXT
    return "[ERROR] " + "; ".join(r["reasons"])

def claim_counts(r):
    v = [c["verdict"] for c in r["claims"]]
    return f"{v.count('explicit')} explicit / {v.count('inferred')} inferred / {v.count('unsupported')} unsupported"

stem = QFILE.stem
OUT_XLSX, OUT_AUDIT = f"{stem}_COMPLETED.xlsx", f"{stem}_audit.csv"

if QFILE.suffix.lower() == ".xlsx":
    wb = openpyxl.load_workbook(QFILE)
    for r in results:
        ws = wb[r["sheet"]]
        c = ws.cell(r["row"], r["answer_col"] + 1, cell_text(r))
        c.fill = PatternFill("solid", fgColor=FILL[r["status"]])
        c.alignment = Alignment(wrap_text=True, vertical="top")
        if r["evidence_col"] is not None and r["source_titles"]:
            ws.cell(r["row"], r["evidence_col"] + 1, "; ".join(r["source_titles"]))
    if "Responder Summary" in wb.sheetnames:
        del wb["Responder Summary"]
    ws = wb.create_sheet("Responder Summary")
    cols = ["Sheet", "Ref", "Question", "Specialist", "Confidence", "Status", "Answer / draft",
            "Sources", "Reasons", "Claims", "Reviewer", "Reviewer decision"]
    ws.append(cols)
    for r in results:
        ws.append([r["sheet"], r["ref"], r["question"], r["category"], r["confidence"], r["status"],
                   r["answer"] if r["status"] != "human_review" else r["draft_answer"],
                   "; ".join(r["source_titles"]), "; ".join(r["reasons"]), claim_counts(r), "", ""])
        ws.cell(ws.max_row, 6).fill = PatternFill("solid", fgColor=FILL[r["status"]])
    for i, w in enumerate([18, 9, 60, 14, 11, 13, 70, 40, 50, 30, 14, 16]):
        ws.column_dimensions[openpyxl.utils.get_column_letter(i + 1)].width = w
    for row in ws.iter_rows(min_row=1):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
            c.font = Font(name="Arial", size=9, bold=(c.row == 1))
    ws.freeze_panes = "D2"
    ws.auto_filter.ref = ws.dimensions
    wb.save(OUT_XLSX)
else:                                                    # CSV in -> CSV out with answers added
    OUT_XLSX = f"{stem}_COMPLETED.csv"
    rows = list(csv.reader(open(QFILE, newline="", encoding="utf-8-sig")))
    by_row = {r["row"]: r for r in results}
    for i, row in enumerate(rows, start=1):
        if i in by_row:
            r = by_row[i]
            row += [""] * (r["answer_col"] + 1 - len(row))
            row[r["answer_col"]] = cell_text(r)
            row += [r["confidence"], r["status"], "; ".join(r["source_titles"])]
    with open(OUT_XLSX, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)

audit = pd.DataFrame([{
    "sheet": r["sheet"], "ref": r["ref"], "row": r["row"], "question": r["question"],
    "question_sent_to_model": r["model_question"], "specialist": r["category"],
    "confidence": r["confidence"], "status": r["status"], "answer": r["answer"],
    "draft_answer": r["draft_answer"], "sources": "; ".join(r["sources"]),
    "reasons": " | ".join(r["reasons"]), "signals": json.dumps(r["signals"]),
    "claims": json.dumps(r["claims"]), "searches": " | ".join(r["search_log"]),
    "seconds": r["seconds"], "reviewer": "", "reviewer_decision": ""} for r in results])
audit.to_csv(OUT_AUDIT, index=False)
print("Saved:", OUT_XLSX, "and", OUT_AUDIT)
files.download(OUT_XLSX)
files.download(OUT_AUDIT)

## 18. (Optional) Check against the answer key
If `golden_set.csv` was uploaded (or is in the pack), this compares each result with what *should* have happened:
- **high** should be auto-approved;
- **medium** should go to human review;
- **partial** and **restricted** should go to a human (review or escalate);
- **gap** should be escalated.

It's a preview of the Phase 4 evals.

In [ ]:
gold_files = sorted(glob.glob("**/golden_set*.csv", recursive=True))
if not gold_files:
    print("No golden_set.csv found; skipping.")
else:
    gold = pd.read_csv(gold_files[0], dtype=str).fillna("")
    EXPECT = {"high": {"auto_approve"}, "medium": {"human_review"}, "gap": {"escalate"},
              "partial": {"human_review", "escalate"},      # either is safe: a human sees it
              "restricted": {"human_review", "escalate"}}
    m = df.merge(gold, on=["sheet", "ref"], how="inner", suffixes=("", "_gold"))
    m["status_ok"] = [s in EXPECT[e] for s, e in zip(m["status"], m["expected_outcome"])]
    m["routing_ok"] = [c in e.split("|") for c, e in zip(m["category"], m["expected_category"])]
    m["sources_ok"] = [not e or bool(set(e.split(";")) & set(s)) for e, s in zip(m["expected_sources"], m["sources"])]
    print(f"Matched {len(m)} of {len(df)} results to the answer key.\n")
    print(f"Outcome correct:  {m['status_ok'].mean():.0%}")
    print(f"Routing correct:  {m['routing_ok'].mean():.0%}")
    print(f"Cites the expected section (where one exists): {m.loc[m['expected_sources'] != '', 'sources_ok'].mean():.0%}\n")
    print("Expected outcome vs what happened:")
    print(pd.crosstab(m["expected_outcome"], m["status"]).to_string())
    misses = m[~m["status_ok"]][["sheet", "ref", "question", "expected_outcome", "status", "confidence", "reasons"]]
    print(f"\n{len(misses)} mismatches:")
    display(misses)